# Leakage-Aware Customer Churn Prediction in Subscription Publishing

Analysis code for the article *"Leakage-Aware Customer Churn Prediction in Subscription Publishing Using the CRISP-DM
Methodology: A Cohort Design and Benchmark of Sixteen Models"* (MDPI *Analytics*).

This notebook reproduces every step of the study, in the order of the CRISP-DM phases:

| Section in this notebook | What it does | Where in the article |
|---|---|---|
| 1. Setup | Libraries, settings and the fixed random seed | Section 2.10 |
| 2. Data understanding | Loads the subscription and invoice extracts | Section 2.2 |
| 3. Data preparation | Builds the cohort, the 12-month target and the 30 features | Sections 2.3–2.5, Table 2 |
| 4. Cohort characteristics | Churn rate by customer group | Section 3.1, Table 4 |
| 5. Modelling setup | Customer-grouped folds, preprocessing inside folds, model settings | Sections 2.6–2.7, Table 3 |
| 6. Benchmark | Nested cross-validation of sixteen approaches | Section 2.7, Figure 2 |
| 7. Evaluation | AUC-PR, ROC-AUC, precision and recall at capacity, bootstrap intervals | Sections 2.8, 3.2–3.3, Table 5, Figure 3 |
| 8. Robustness check | Same design at a second prediction date | Sections 2.9, 3.4, Table 6 |
| 9. Leak test | What happens when undated attributes are added back | Sections 2.9, 3.5, Table 7, Figure 4 |
| 10. Timing of cancellations | When churners leave within the 12-month window | Section 4.1.4 |

**Data.** The data are commercially confidential and are **not** included. The notebook expects two CSV files in a
folder `data/` (see Section 1). Nothing in this notebook prints individual customer records; only counts and
aggregated results are shown.

**One principle runs through the whole notebook:** every prediction is made from a fixed date, using only
information recorded *before* that date. The outcome is observed only *after* it.

## 1. Setup

### 1.1 Libraries
The study uses `pandas` and `numpy` for data handling, `scikit-learn` for most models and for cross-validation,
`xgboost`, `lightgbm` (tuned with `optuna`) and `catboost` for gradient boosting, `interpret` for the Explainable
Boosting Machine, and `scikit-survival` for the two survival models.

Install with:
```
pip install pandas numpy scikit-learn xgboost lightgbm catboost optuna interpret-core scikit-survival matplotlib
```

**Note on versions.** The seed fixes the folds, the tuning and the bootstrap, so logistic regression, the tenure rule,
the robustness check and the leak test give the same numbers on every installation. Some models (in particular the
neural network, random forest, XGBoost, the EBM and the random survival forest) can give slightly different scores
with other library versions or hardware. The versions used are listed in Section 2.10 of the article; the next cell prints those of your installation.

In [ ]:
import os, time, warnings
os.environ['PYTHONHASHSEED'] = '42'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
import lightgbm as lgb
import optuna
from catboost import CatBoostClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.width', 200)

import sklearn, xgboost, catboost, interpret, sksurv
for m in [pd, np, sklearn, xgboost, lgb, catboost, optuna, interpret, sksurv]:
    print(f'{m.__name__:12s} {m.__version__}')

### 1.2 Settings
All random processes use the same seed (42), so the folds, the tuning and the bootstrap are identical on every run.

* `PREDICTION_DATE` is the main prediction date (1 July 2024). It is the latest date that leaves a full 12-month
  outcome window before the extracts end on 30 June 2025.
* `ROBUSTNESS_DATE` is the second prediction date (1 January 2024), used only for the robustness check.
* `MIGRATION_DATE` is a placeholder start date (25 December 2018) that marks records migrated from a previous
  service provider.

In [ ]:
SEED = 42
np.random.seed(SEED)

DATA_DIR = 'data/'                      # folder that holds the two extracts
SUBSCRIPTION_FILE = 'preprocessed.csv'  # subscription extract (one row per subscription)
INVOICE_FILE = 'invoices.csv'           # invoice extract (one row per invoice)
OUT_DIR = 'results/'
os.makedirs(OUT_DIR, exist_ok=True)

PREDICTION_DATE = '2024-07-01'
ROBUSTNESS_DATE = '2024-01-01'
HORIZON_MONTHS = 12
MIGRATION_DATE = pd.Timestamp('2018-12-25')

## 2. Data understanding

### 2.1 Load the two extracts
The extracts were exported with a first line `sep=,` (an Excel convention). The helper below skips that line when it
is present. Date columns are converted to real dates straight away, because every later step compares dates.

In [ ]:
def read_extract(path):
    with open(path, encoding='utf-8') as f:
        first = f.readline()
    return pd.read_csv(path, skiprows=1 if first.startswith('sep=') else 0, low_memory=False)

subs = read_extract(DATA_DIR + SUBSCRIPTION_FILE)
inv = read_extract(DATA_DIR + INVOICE_FILE)

for c in ['subscription_start_date', 'subscription_cancellation_date', 'subscription_entry_date', 'recipient_entry_date']:
    subs[c] = pd.to_datetime(subs[c])
for c in ['invoice_date', 'payment_date']:
    inv[c] = pd.to_datetime(inv[c])

print('Subscription extract:', subs.shape[0], 'subscriptions,', subs.recipient_customer_nr.nunique(), 'customers,',
      subs.shape[1], 'columns')
print('Invoice extract:     ', inv.shape[0], 'invoices,', inv.shape[1], 'columns')

### 2.2 Date ranges
A quick check of the period each extract covers. The invoice history starts in August 2023, so invoice features
before the main prediction date cover about ten months.

In [ ]:
cancel = subs.subscription_cancellation_date.dropna()
print('Cancellation dates:', cancel.min().date(), 'to', cancel.max().date(), f'({len(cancel)} cancelled subscriptions)')
print('Cancellation dates after the extract date (19 June 2025):', int((cancel > '2025-06-19').sum()))
print('Invoice dates:     ', inv.invoice_date.min().date(), 'to', inv.invoice_date.max().date())
print('Share of subscriptions with the migration placeholder as start date:',
      round(subs.subscription_start_date.eq(MIGRATION_DATE).mean(), 3))

### 2.3 Real start date
For migrated records, the start date is the placeholder 25 December 2018, not the real start of the subscription.
For these records the entry date (`subscription_entry_date`) is used as the real start, so that tenure is measured correctly.

Invoices are linked to subscriptions through the subscription number; invoices without one are dropped.

In [ ]:
subs['start_migrated'] = subs.subscription_start_date.eq(MIGRATION_DATE)
subs['start_real'] = pd.to_datetime(np.where(subs.start_migrated, subs.subscription_entry_date, subs.subscription_start_date))

inv = inv.dropna(subset=['invoice_subscription_nr']).copy()
inv['invoice_subscription_nr'] = inv.invoice_subscription_nr.astype('int64')
inv = inv.sort_values(['invoice_subscription_nr', 'invoice_date', 'invoice_nr'])

print('Invoices linked to a subscription:', len(inv))
print('Subscriptions in the extract with at least one invoice:', int(subs.subscription_nr.isin(inv.invoice_subscription_nr).sum()))

## 3. Data preparation

The dataset is built as a **cross-sectional cohort**: one row per subscription that is active on the prediction date.
The steps below are written as small functions, so the same code builds both the main cohort and the robustness
cohort.

### 3.1 Cohort and target
* **Cohort:** subscriptions that started before the prediction date and had not been cancelled before it.
* **Target (`target_churn_12m`):** 1 if the cancellation date falls in the 12 months after the prediction date,
  0 otherwise.
* **Survival outcome:** days from the prediction date to cancellation, censored at the end of the window. This is
  used only by the two survival models, never as a feature.

In [ ]:
def select_cohort(C):
    active = (subs.start_real < C) & (subs.subscription_cancellation_date.isna() | (subs.subscription_cancellation_date >= C))
    return subs[active].copy()


def make_target(s, C, E):
    out = pd.DataFrame({'subscription_id': s.subscription_nr.values,
                        'customer_id': s.recipient_customer_nr.values,
                        'cutoff_date': C.date()})
    cancel = s.subscription_cancellation_date.values
    out['target_churn_12m'] = ((cancel >= C) & (cancel < E)).astype(int)
    out['outcome_event_in_window'] = out.target_churn_12m
    days = (pd.Series(cancel) - C).dt.days.values
    out['outcome_days_to_event'] = np.where(out.target_churn_12m == 1, days, (E - C).days)
    return out

### 3.2 Customer and offer features
These attributes describe the customer and the subscription as recorded at sign-up: gender, province, age group,
offer code, origin code and sign-up method. Tenure is the number of days from the real start date to the
prediction date.

In [ ]:
def add_static_features(out, s, C):
    out['gender'] = s.recipient_gender.values
    out['province'] = s.recipient_province.values
    out['age_group'] = s.recipient_age_group.values
    out['offer_code'] = s.subscription_offer_code.values
    out['origin_code'] = s.subscription_origin_code.values
    out['signup_method'] = s.subscription_signup_method.values
    out['tenure_days'] = (C - s.start_real).dt.days.values
    out['start_migrated'] = s.start_migrated.astype(int).values
    return out

### 3.3 Customer history
A customer can hold more than one subscription. For each customer we count, using only events before the
prediction date, how many subscriptions were started, how many were cancelled, and how many are active.

In [ ]:
def add_customer_history(out, s, C):
    allc = subs[subs.recipient_customer_nr.isin(s.recipient_customer_nr)]
    started = allc[allc.start_real < C].groupby('recipient_customer_nr').size()
    ended = allc[allc.subscription_cancellation_date < C].groupby('recipient_customer_nr').size()
    active = allc[(allc.start_real < C) & (allc.subscription_cancellation_date.isna() | (allc.subscription_cancellation_date >= C))] \
        .groupby('recipient_customer_nr').size()
    k = s.recipient_customer_nr
    out['cust_n_subs_started_before'] = k.map(started).fillna(0).astype(int).values
    out['cust_n_subs_cancelled_before'] = k.map(ended).fillna(0).astype(int).values
    out['cust_n_active_subs'] = k.map(active).fillna(0).astype(int).values
    return out

### 3.4 Invoice and payment features
Only invoices dated **before** the prediction date are used. Two rules keep these features free of later
information:

1. An invoice counts as **paid** only if its payment date is before the prediction date; otherwise it is treated as
   unpaid on that date.
2. **Payment reminders** are counted only for invoices settled before the prediction date, because the reminder count
   in the extract may include reminders sent later.

Subscriptions without any invoice before the prediction date get empty values; these are handled inside each
training fold (Section 5.2).

In [ ]:
def add_invoice_features(out, s, C):
    iv = inv[(inv.invoice_date < C) & inv.invoice_subscription_nr.isin(s.subscription_nr)].copy()
    iv['paid_by_C'] = iv.payment_date.notna() & (iv.payment_date < C)
    iv['delay'] = np.where(iv.paid_by_C, (iv.payment_date - iv.invoice_date).dt.days, np.nan)
    iv['late30'] = (iv.delay > 30).astype(float)
    iv['open_amt'] = np.where(iv.paid_by_C, 0.0, iv.gross_amount)
    iv['rem_resolved'] = np.where(iv.paid_by_C, iv.n_reminders, 0)

    g = iv.groupby('invoice_subscription_nr')
    last = g.tail(1).set_index('invoice_subscription_nr')
    prev = g.nth(-2).set_index('invoice_subscription_nr')
    f = pd.DataFrame(index=g.size().index)
    f['inv_n_invoices'] = g.size()
    f['inv_days_since_last_invoice'] = (C - last.invoice_date).dt.days
    f['inv_last_amount'] = last.gross_amount
    f['inv_mean_amount'] = g.gross_amount.mean()
    f['inv_price_change_pct'] = ((last.gross_amount - prev.gross_amount) / prev.gross_amount.replace(0, np.nan) * 100).reindex(f.index)
    f['inv_mean_pay_delay_days'] = g.delay.mean()
    f['inv_max_pay_delay_days'] = g.delay.max()
    f['inv_n_paid_late_30d'] = g.late30.sum()
    f['inv_n_unpaid_at_cutoff'] = g.paid_by_C.apply(lambda x: (~x).sum())
    f['inv_open_amount_at_cutoff'] = g.open_amt.sum()
    f['inv_n_reminders'] = g.rem_resolved.sum()
    f['inv_payment_method'] = last.payment_method

    out = out.merge(f, left_on='subscription_id', right_index=True, how='left')
    out['has_invoice_history'] = out.inv_n_invoices.notna().astype(int)
    out['inv_n_invoices'] = out.inv_n_invoices.fillna(0).astype(int)
    out['inv_payment_method'] = out.inv_payment_method.fillna('None')
    return out

### 3.5 Undated attributes (leak test only)
Four attributes in the subscription extract have no date: delivery complaints, service issues per year and two
contact scores. They may include events after the cancellation, so they are **not** used in the main models. They
are kept in separate columns (prefix `undated_`) only for the leak test in Section 9.

All other attributes that reflect the cancellation itself (for example, the amount of the most recent invoice, which
is zero once billing stops, or the stored time-to-event field) are simply never copied into the dataset.

In [ ]:
UNDATED_SOURCES = [('stats_has_delivery_complaint', 'undated_delivery_complaint'),
                   ('stats_issues_per_year', 'undated_issues_per_year'),
                   ('recipient_reachability_score', 'undated_reachability_score'),
                   ('recipient_contact_score', 'undated_contact_score')]


def add_undated(out, s):
    for src, dst in UNDATED_SOURCES:
        out[dst] = s[src].astype(float).values if s[src].dtype == bool else s[src].values
    return out

### 3.6 Build the two cohorts
The function `build_cohort` chains the steps above. It is run once for the main prediction date and once for the
robustness date.

In [ ]:
def build_cohort(date, months=HORIZON_MONTHS):
    C = pd.Timestamp(date)
    E = C + pd.DateOffset(months=months)
    s = select_cohort(C)
    out = make_target(s, C, E)
    out = add_static_features(out, s, C)
    out = add_customer_history(out, s, C)
    out = add_invoice_features(out, s, C)
    out = add_undated(out, s)
    return out.reset_index(drop=True)


main_raw = build_cohort(PREDICTION_DATE)
rob_raw = build_cohort(ROBUSTNESS_DATE)

for name, d in [('Main cohort (1 July 2024)', main_raw), ('Robustness cohort (1 January 2024)', rob_raw)]:
    print(f'{name}: {len(d)} subscriptions, {d.customer_id.nunique()} customers, '
          f'{d.target_churn_12m.sum()} churners ({d.target_churn_12m.mean():.1%})')

shared = set(main_raw.subscription_id) & set(rob_raw.subscription_id)
print(f'Subscriptions in both cohorts: {len(shared)} ({len(shared) / len(main_raw):.0%} of the main cohort)')

both = main_raw.merge(rob_raw, on='subscription_id', suffixes=('_main', '_rob'))
print('Churners in both cohorts:', int(((both.target_churn_12m_main == 1) & (both.target_churn_12m_rob == 1)).sum()))

### 3.7 Feature list
The models use **30 features**: 7 categorical and 23 numeric.

* The 17 numeric features built above, plus six derived flags: gender recorded, log of tenure, tenure under one
  year, tenure under two years, price increase on the last invoice, and non-standard offer code.
* Gender has two codes for female (`F` and the Dutch `V` for *vrouw*); both are merged into `F`.
* Missing categories are coded as `'Missing'`.

Identifiers (`subscription_id`, `customer_id`), the target and the survival outcome are never given to a model.
`customer_id` is used only to keep each customer's subscriptions together in the same fold.

In [ ]:
CAT = ['gender', 'province', 'age_group', 'offer_code', 'origin_code', 'signup_method', 'inv_payment_method']
NUM = ['tenure_days', 'start_migrated', 'cust_n_subs_started_before', 'cust_n_subs_cancelled_before',
       'cust_n_active_subs', 'inv_n_invoices', 'inv_days_since_last_invoice', 'inv_last_amount', 'inv_mean_amount',
       'inv_price_change_pct', 'inv_mean_pay_delay_days', 'inv_max_pay_delay_days', 'inv_n_paid_late_30d',
       'inv_n_unpaid_at_cutoff', 'inv_open_amount_at_cutoff', 'inv_n_reminders', 'has_invoice_history']
UNDATED = [dst for _, dst in UNDATED_SOURCES]
STANDARD_OFFERS = ['HJ', 'JR', 'NRM']   # standard (half-)yearly and normal tariffs


def fill_categories(d):
    d = d.copy()
    for c in CAT:
        d[c] = d[c].fillna('Missing').astype(str)
    return d


def add_derived_flags(d):
    d = d.copy()
    d['gender'] = d.gender.replace({'V': 'F'})
    d['gender_known'] = (d.gender != 'O').astype(int)
    d['log_tenure'] = np.log1p(d.tenure_days)
    d['tenure_lt_1y'] = (d.tenure_days < 365).astype(int)
    d['tenure_lt_2y'] = (d.tenure_days < 730).astype(int)
    d['price_up'] = (d.inv_price_change_pct.fillna(0) > 0).astype(int)
    d['offer_intro'] = (~d.offer_code.isin(STANDARD_OFFERS)).astype(int)
    return d


NUMF = NUM + ['gender_known', 'log_tenure', 'tenure_lt_1y', 'tenure_lt_2y', 'price_up', 'offer_intro']
FEAT = NUMF + CAT

main = add_derived_flags(fill_categories(main_raw))
print('Number of features:', len(FEAT), f'({len(CAT)} categorical, {len(NUMF)} numeric)')

## 4. Cohort characteristics (Table 4)
Churn rate within customer groups of the main cohort. This is a descriptive view of the data before any modelling.

In [ ]:
t = main.tenure_days
groups = {
    'Tenure under 1 year': t < 365,
    'Tenure 1 to 2 years': (t >= 365) & (t < 730),
    'Tenure 2 to 5 years': (t >= 730) & (t < 1826),
    'Tenure 5 to 10 years': (t >= 1826) & (t < 3652),
    'Tenure over 10 years': t >= 3652,
    'Entered after the migration': main.start_migrated == 0,
    'Migrated record': main.start_migrated == 1,
    'Non-standard offer code': main.offer_intro == 1,
    'Standard offer code': main.offer_intro == 0,
    'Sign-up method recorded': main.signup_method != 'Onbekend',
    'Sign-up method unknown': main.signup_method == 'Onbekend',
    'Earlier cancellation by the same customer': main.cust_n_subs_cancelled_before > 0,
    'No earlier cancellation': main.cust_n_subs_cancelled_before == 0,
    'Price increase on the last invoice': main.price_up == 1,
    'No price increase': main.price_up == 0,
    'Paid an invoice more than 30 days late': main.inv_n_paid_late_30d > 0,
    'Never paid more than 30 days late': main.inv_n_paid_late_30d.fillna(0) == 0,
    'All subscriptions': pd.Series(True, index=main.index),
}
table4 = pd.DataFrame([{'Group': k, 'Subscriptions': int(m.sum()), 'Churned': int(main.target_churn_12m[m].sum()),
                        'Churn rate': f'{main.target_churn_12m[m].mean():.1%}'} for k, m in groups.items()])
table4

## 5. Modelling setup

### 5.1 Customer-grouped folds
The cohort is split into **five test groups** with `StratifiedGroupKFold`:

* **Grouped:** all subscriptions of the same customer fall in the same group, so a customer never appears in both
  training and test data.
* **Stratified:** each group has about the same share of churners (5.1%).

Each subscription is scored exactly once, by a model that was trained without it. Inside each training part, a
second split into three folds (also grouped and stratified) is used for tuning. This is **nested cross-validation**
(Figure 2 in the article).

In [ ]:
y = main.target_churn_12m.values
groups_id = main.customer_id.values
outer = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(main, y, groups_id))

for k, (tr, te) in enumerate(outer, 1):
    print(f'Test group {k}: {len(te)} subscriptions, churn rate {y[te].mean():.3f}')

### 5.2 Preprocessing inside each fold
Every step that learns something from the data is fitted on the **training part of each fold only**, then applied to
the test part:

* **Rare categories:** category values that occur fewer than 30 times in the training part are grouped as `'Other'`.
* **Missing numbers:** replaced by the training median, with an extra 0/1 column that marks the missing value.
* **Scaling** of numeric features and **one-hot encoding** of categories.

Class imbalance is handled with **class weights** (churners count more in the loss), not with synthetic oversampling
such as SMOTE.

In [ ]:
def rare_group(tr, te, cols, min_n=30):
    tr, te = tr.copy(), te.copy()
    for c in cols:
        counts = tr[c].value_counts()
        keep = set(counts[counts >= min_n].index)
        tr[c] = tr[c].where(tr[c].isin(keep), 'Other')
        te[c] = te[c].where(te[c].isin(keep), 'Other')
    return tr, te


def prep(num=NUMF):
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)), ('sc', StandardScaler())]), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT)])


SPW = float(np.sqrt((1 - y.mean()) / y.mean()))   # moderate up-weighting of churners for XGBoost (about 4.3)

### 5.3 Models tuned with a grid search (Table 3)
Nine models are tuned with a small grid search on the inner three folds, scored by AUC-PR. The values in braces are
the candidates; the inner cross-validation picks one per fold.

In [ ]:
def tuned(model, grid):
    return Pipeline([('prep', prep()), ('m', model)]), {f'm__{k}': v for k, v in grid.items()}


SK_MODELS = {
    'Logistic regression': tuned(LogisticRegression(class_weight='balanced', max_iter=5000, random_state=SEED),
                                 {'C': [0.01, 0.1, 1.0]}),
    'Naive Bayes': tuned(GaussianNB(), {'var_smoothing': [1e-9, 1e-6, 1e-3, 1e-1]}),
    'Decision tree': tuned(DecisionTreeClassifier(class_weight='balanced', random_state=SEED),
                           {'max_depth': [3, 5, 7], 'min_samples_leaf': [50, 150]}),
    'k-nearest neighbours': tuned(KNeighborsClassifier(), {'n_neighbors': [50, 150, 300]}),
    'SVM (RBF kernel)': tuned(SVC(kernel='rbf', class_weight='balanced', random_state=SEED),
                              {'C': [0.1, 1.0], 'gamma': ['scale']}),
    'Random forest': tuned(RandomForestClassifier(n_estimators=500, max_features=0.3, class_weight='balanced_subsample',
                                                  n_jobs=-1, random_state=SEED), {'min_samples_leaf': [10, 25, 50]}),
    'AdaBoost': tuned(AdaBoostClassifier(random_state=SEED), {'n_estimators': [100, 300], 'learning_rate': [0.1, 0.5]}),
    'XGBoost': tuned(XGBClassifier(subsample=0.8, colsample_bytree=0.8, learning_rate=0.05, eval_metric='aucpr',
                                   n_jobs=4, random_state=SEED, verbosity=0),
                     {'max_depth': [2, 3, 4], 'n_estimators': [200, 400], 'scale_pos_weight': [1.0, SPW]}),
    'Neural network (MLP)': tuned(MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, max_iter=500,
                                                random_state=SEED), {'alpha': [1e-4, 1e-2, 1.0]}),
}

### 5.4 LightGBM, tuned with Optuna
LightGBM handles categories natively. Its settings are tuned with Optuna (20 trials per fold) on the inner three
folds, with early stopping on each inner validation part; the average number of boosting rounds is then used to
refit on the full training part.

In [ ]:
def lgb_prep(X):
    X = X.copy()
    for c in CAT:
        X[c] = X[c].astype('category')
    return X


def lgb_params(p):
    return dict(objective='binary', learning_rate=p['lr'], num_leaves=p['num_leaves'], min_child_samples=p['mcs'],
                feature_fraction=p['ff'], bagging_fraction=p['bf'], bagging_freq=1, lambda_l2=p['l2'],
                cat_smooth=p['cs'], scale_pos_weight=p['spw'], verbose=-1, seed=SEED)


def tune_lgb(Xtr, ytr, gtr, num, n_trials):
    inner = list(StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(Xtr, ytr, gtr))
    X = lgb_prep(Xtr[num + CAT])

    def objective(t):
        p = lgb_params({'lr': t.suggest_float('lr', 0.01, 0.1, log=True), 'num_leaves': t.suggest_int('num_leaves', 4, 48),
                        'mcs': t.suggest_int('mcs', 20, 200), 'ff': t.suggest_float('ff', 0.5, 1.0),
                        'bf': t.suggest_float('bf', 0.5, 1.0), 'l2': t.suggest_float('l2', 1e-3, 30, log=True),
                        'cs': t.suggest_float('cs', 1, 50), 'spw': t.suggest_float('spw', 1, 19)})
        scores, rounds = [], []
        for a, b in inner:
            m = lgb.train(p, lgb.Dataset(X.iloc[a], ytr[a]), 2000, valid_sets=[lgb.Dataset(X.iloc[b], ytr[b])],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
            scores.append(average_precision_score(ytr[b], m.predict(X.iloc[b])))
            rounds.append(m.best_iteration)
        t.set_user_attr('n', int(np.mean(rounds)))
        return np.mean(scores)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    return lgb_params(study.best_trial.params), max(study.best_trial.user_attrs['n'], 50)

### 5.5 CatBoost
CatBoost also handles categories natively. The number of trees is chosen by early stopping on a grouped hold-out part
of the training data; the model is then refitted on the full training part with that number of trees. Class weights
use the square root of the class ratio (`SqrtBalanced`).

In [ ]:
def fit_cat(Xtr, ytr, gtr, Xte, num):
    a, b = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(Xtr, ytr, gtr))
    cols = num + CAT
    settings = dict(learning_rate=0.03, depth=5, l2_leaf_reg=5, auto_class_weights='SqrtBalanced',
                    random_seed=SEED, verbose=0, thread_count=4)
    m = CatBoostClassifier(iterations=3000, eval_metric='PRAUC', od_type='Iter', od_wait=200, **settings)
    m.fit(Xtr.iloc[a][cols], ytr[a], cat_features=CAT, eval_set=(Xtr.iloc[b][cols], ytr[b]))
    n = max(m.get_best_iteration(), 100)
    m2 = CatBoostClassifier(iterations=n, **settings)
    m2.fit(Xtr[cols], ytr, cat_features=CAT)
    return m2.predict_proba(Xte[cols])[:, 1], m2

## 6. Benchmark: nested cross-validation

### 6.1 Train and score every model on every fold
For each of the five test groups, every model is trained on the other four groups and scores the held-out group.
The out-of-fold scores are stored in `oof`. Besides the grid-search models, LightGBM and CatBoost, the loop fits:

* the **Explainable Boosting Machine** (EBM), a glass-box model that shows the effect of each feature;
* a **Cox survival model** (elastic-net penalised) and a **random survival forest**, which use the time to
  cancellation within the window.

This cell takes the longest to run (roughly 20 to 60 minutes, depending on the machine).

In [ ]:
ALL = list(SK_MODELS) + ['LightGBM', 'CatBoost', 'Explainable Boosting Machine',
                          'Cox survival model', 'Random survival forest']
oof = {k: np.zeros(len(main)) for k in ALL}
best_params = {k: [] for k in ALL}

for f, (tr, te) in enumerate(outer):
    start = time.time()
    Xtr, Xte = rare_group(main.iloc[tr], main.iloc[te], CAT)
    Xtr = Xtr.reset_index(drop=True)
    ytr, gtr = y[tr], groups_id[tr]
    inner = list(StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(Xtr, ytr, gtr))

    # grid-search models
    for name, (pipe, grid) in SK_MODELS.items():
        gs = GridSearchCV(pipe, grid, scoring='average_precision', cv=inner,
                          n_jobs=1 if name == 'Random forest' else -1, refit=True)
        gs.fit(Xtr[FEAT], ytr)
        est = gs.best_estimator_
        oof[name][te] = (est.decision_function(Xte[FEAT]) if name == 'SVM (RBF kernel)'
                         else est.predict_proba(Xte[FEAT])[:, 1])
        best_params[name].append({k.replace('m__', ''): v for k, v in gs.best_params_.items()})

    # LightGBM
    p, n = tune_lgb(Xtr, ytr, gtr, NUMF, 20)
    oof['LightGBM'][te] = lgb.train({**p, 'deterministic': True}, lgb.Dataset(lgb_prep(Xtr[FEAT]), ytr), n) \
        .predict(lgb_prep(Xte[FEAT]))
    best_params['LightGBM'].append({'num_leaves': p['num_leaves'], 'learning_rate': round(p['learning_rate'], 3), 'rounds': n})

    # CatBoost
    oof['CatBoost'][te], cm = fit_cat(Xtr, ytr, gtr, Xte, NUMF)
    best_params['CatBoost'].append({'iterations': cm.tree_count_})

    # Explainable Boosting Machine
    ebm = ExplainableBoostingClassifier(interactions=10, outer_bags=8, random_state=SEED, n_jobs=-1)
    oof['Explainable Boosting Machine'][te] = ebm.fit(Xtr[FEAT], ytr).predict_proba(Xte[FEAT])[:, 1]

    # survival models (preprocessing fitted on the training part only)
    pre = prep().fit(Xtr[FEAT])
    A, B = pre.transform(Xtr[FEAT]), pre.transform(Xte[FEAT])
    ys = Surv.from_arrays(Xtr.outcome_event_in_window.astype(bool).values,
                          np.maximum(Xtr.outcome_days_to_event.values, 1))
    cox = CoxnetSurvivalAnalysis(l1_ratio=0.5, alpha_min_ratio=0.05, n_alphas=30, fit_baseline_model=False).fit(A, ys)
    oof['Cox survival model'][te] = cox.predict(B, alpha=cox.alphas_[len(cox.alphas_) // 2])
    rsf = RandomSurvivalForest(300, min_samples_leaf=30, max_features=0.3, n_jobs=-1, random_state=SEED, low_memory=True)
    oof['Random survival forest'][te] = rsf.fit(A, ys).predict(B)

    print(f'Test group {f + 1} of 5 scored in {time.time() - start:.0f} s', flush=True)

### 6.2 Ranks, ensemble and the tenure-only rule
* The SVM and the survival models return scores that are not probabilities (a margin or a risk score). They are
  converted to ranks within each test group, which leaves all ranking metrics unchanged.
* The **ensemble** averages the ranks of logistic regression, CatBoost and the EBM within each test group.
* The **tenure-only rule** needs no training: it ranks subscriptions from the shortest to the longest tenure.

In [ ]:
for k in ['SVM (RBF kernel)', 'Cox survival model', 'Random survival forest']:
    for _, te in outer:
        oof[k][te] = rankdata(oof[k][te]) / (len(te) + 1)

ENSEMBLE = 'Ensemble (LR + CatBoost + EBM)'
oof[ENSEMBLE] = np.zeros(len(main))
for _, te in outer:
    oof[ENSEMBLE][te] = np.mean([rankdata(oof[k][te]) / len(te)
                                 for k in ['Logistic regression', 'CatBoost', 'Explainable Boosting Machine']], axis=0)

oof['Tenure-only rule'] = -main.tenure_days.values.astype(float)

pd.DataFrame(best_params['Logistic regression'], index=[f'Fold {i}' for i in range(1, 6)])

## 7. Evaluation

### 7.1 Metrics
* **AUC-PR** (average precision) is the primary metric. With 5.1% churners, random guessing scores 0.051.
* **ROC-AUC** is reported for comparison with earlier studies.
* **Precision in the top k%**: the share of churners among the k% of subscriptions with the highest predicted risk.
  This matches how a retention team works: it can contact only a limited number of customers.
* **Recall in the top 10%**: the share of all churners that fall in the top 10%.

Every metric is computed **per test group** and then averaged over the five groups (AUC-PR and ROC-AUC also with
their standard deviation). Computing metrics per group, rather than on all scores pooled together, matters for models
whose score scale differs from one fold to the next, such as XGBoost.

In [ ]:
test_sets = [te for _, te in outer]


def precision_top(yt, p, q):
    k = int(round(q * len(yt)))
    return yt[np.argsort(-p, kind='stable')[:k]].mean()


def recall_top(yt, p, q):
    k = int(round(q * len(yt)))
    return yt[np.argsort(-p, kind='stable')[:k]].sum() / yt.sum()


def per_group(metric, p, idx_list=test_sets, **kw):
    return np.array([metric(y[i], p[i], **kw) for i in idx_list])

### 7.2 Performance table (Table 5)

In [ ]:
rows = []
for k, p in oof.items():
    ap_, roc_ = per_group(average_precision_score, p), per_group(roc_auc_score, p)
    rows.append({'Approach': k,
                 'AUC-PR': f'{ap_.mean():.3f} ± {ap_.std():.3f}',
                 'ROC-AUC': f'{roc_.mean():.3f} ± {roc_.std():.3f}',
                 'Precision top 5%': f'{per_group(precision_top, p, q=0.05).mean():.1%}',
                 'Precision top 10%': f'{per_group(precision_top, p, q=0.10).mean():.1%}',
                 'Recall top 10%': f'{per_group(recall_top, p, q=0.10).mean():.1%}',
                 '_sort': ap_.mean()})
table5 = pd.DataFrame(rows).sort_values('_sort', ascending=False).drop(columns='_sort').reset_index(drop=True)
table5.loc[len(table5)] = ['Random guessing', f'{y.mean():.3f}', '0.500', f'{y.mean():.1%}', f'{y.mean():.1%}', '10.0%']
table5

### 7.3 Is any model better than logistic regression? (paired bootstrap)
To test whether a model differs from logistic regression, we use a **paired bootstrap drawn within each test group**:

1. In each of 1,000 rounds, subscriptions are resampled with replacement **within each test group**.
2. AUC-PR is computed per group and **averaged over the five groups**, exactly as in Table 5.
3. The difference with logistic regression is taken on the same resample (paired).

The 2.5th and 97.5th percentiles of the 1,000 differences form a 95% interval. An interval that includes zero means
the model is not reliably different from logistic regression.

In [ ]:
rng = np.random.default_rng(SEED)
boot = [[rng.choice(i, len(i)) for i in test_sets] for _ in range(1000)]


def mean_ap(p, idx_list):
    return np.mean([average_precision_score(y[i], p[i]) for i in idx_list])


ref_boot = np.array([mean_ap(oof['Logistic regression'], b) for b in boot])
ref_mean = mean_ap(oof['Logistic regression'], test_sets)
rows = []
for k, p in oof.items():
    diff = np.array([mean_ap(p, b) for b in boot]) - ref_boot
    lo, hi = np.percentile(diff, [2.5, 97.5])
    rows.append({'Approach': k, 'Mean AUC-PR': round(mean_ap(p, test_sets), 3),
                 'Difference vs. LR': round(mean_ap(p, test_sets) - ref_mean, 3),
                 '95% interval': f'{lo:+.3f} to {hi:+.3f}', 'Includes zero': lo <= 0 <= hi})
bootstrap_table = pd.DataFrame(rows).sort_values('Mean AUC-PR', ascending=False).reset_index(drop=True)
bootstrap_table

### 7.4 Tenure-only rule vs. logistic regression at retention capacity
At the capacities a retention team would realistically use, does the model find more churners than simply contacting
the newest customers? The same within-group bootstrap is used for the difference in precision. We also count how many
subscriptions the two top-10% lists have in common.

In [ ]:
lr, ten = oof['Logistic regression'], oof['Tenure-only rule']
rows = []
for q in (0.05, 0.10, 0.20):
    d = [per_group(precision_top, lr, b, q=q).mean() - per_group(precision_top, ten, b, q=q).mean() for b in boot]
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({'Capacity': f'Top {q:.0%}',
                 'Precision LR': f'{per_group(precision_top, lr, q=q).mean():.1%}',
                 'Precision tenure rule': f'{per_group(precision_top, ten, q=q).mean():.1%}',
                 'Difference, 95% interval (percentage points)': f'{100 * lo:+.1f} to {100 * hi:+.1f}'})
display(pd.DataFrame(rows))

overlap = []
for i in test_sets:
    k = int(round(0.10 * len(i)))
    top_lr = set(i[np.argsort(-lr[i], kind='stable')[:k]])
    top_ten = set(i[np.argsort(-ten[i], kind='stable')[:k]])
    overlap.append(len(top_lr & top_ten) / k)
print(f'Share of the top 10% shared by both rankings: {np.mean(overlap):.0%}')

### 7.5 Figure 3: mean AUC-PR of all approaches

In [ ]:
fig_data = pd.DataFrame({'Approach': list(oof),
                         'mean': [per_group(average_precision_score, p).mean() for p in oof.values()],
                         'sd': [per_group(average_precision_score, p).std() for p in oof.values()]}) \
    .sort_values('mean')
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(fig_data.Approach, fig_data['mean'], xerr=fig_data['sd'], color='#4C72B0', capsize=3)
ax.axvline(y.mean(), ls='--', color='grey')
ax.text(y.mean() + 0.002, -0.6, 'Random guessing', color='grey', fontsize=9)
ax.set_xlabel('AUC-PR (mean ± SD over five test groups)')
plt.tight_layout()
plt.savefig(OUT_DIR + 'figure3_auc_pr.png', dpi=200)
plt.show()

## 8. Robustness check: a second prediction date (Table 6)

The same design is repeated with 1 January 2024 as the prediction date. The check uses the **first-round
configuration**, applied identically at both dates: the 17 base features without the derived flags, logistic
regression with fixed settings, LightGBM (25 Optuna trials), CatBoost and the tenure-only rule. Metrics are computed
on the pooled out-of-fold predictions.

Note that the two cohorts share most of their subscriptions (Section 3.6). The check shows whether the results depend
on the choice of date; it is not a test on a later, unseen cohort.

In [ ]:
def fit_lr_first_round(Xtr, ytr, Xte, num):
    pre = ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), num),
                             ('c', OneHotEncoder(handle_unknown='ignore'), CAT)])
    m = make_pipeline(pre, LogisticRegression(C=0.3, class_weight='balanced', max_iter=3000))
    m.fit(Xtr, ytr)
    return m.predict_proba(Xte)[:, 1]


def first_round(d, num, n_trials=25):
    yy, gg = d.target_churn_12m.values, d.customer_id.values
    res = {k: np.zeros(len(d)) for k in ['Logistic regression', 'CatBoost', 'LightGBM', 'Tenure-only rule']}
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(d, yy, gg):
        Xtr, Xte = rare_group(d.iloc[tr], d.iloc[te], CAT)
        Xtr = Xtr.reset_index(drop=True)
        res['Tenure-only rule'][te] = -Xte.tenure_days.values
        res['Logistic regression'][te] = fit_lr_first_round(Xtr, yy[tr], Xte, num)
        p, n = tune_lgb(Xtr, yy[tr], gg[tr], num, n_trials)
        res['LightGBM'][te] = lgb.train(p, lgb.Dataset(lgb_prep(Xtr[num + CAT]), yy[tr]), n).predict(lgb_prep(Xte[num + CAT]))
        res['CatBoost'][te], _ = fit_cat(Xtr, yy[tr], gg[tr], Xte, num)
    return yy, res


cohorts = {'1 July 2024': fill_categories(main_raw), '1 January 2024': fill_categories(rob_raw)}
first = {}
table6 = {}
for label, d in cohorts.items():
    yy, res = first_round(d, NUM)
    first[label] = (yy, res)
    for k, p in res.items():
        table6.setdefault(k, {})[f'AUC-PR, {label}'] = round(average_precision_score(yy, p), 3)
        table6[k][f'ROC-AUC, {label}'] = round(roc_auc_score(yy, p), 3)
    table6.setdefault('Base rate', {})[f'AUC-PR, {label}'] = round(yy.mean(), 3)
    table6['Base rate'][f'ROC-AUC, {label}'] = 0.5
pd.DataFrame(table6).T

## 9. Leak test

### 9.1 Attributes with high predictive power on their own (Table 7)
Each attribute is used alone as a score. An attribute that separates churners almost perfectly on its own is a warning
sign: in a churn setting, that usually means the value was recorded after, or because of, the cancellation. The ROC-AUC
is taken in the direction that gives the higher value.

Three attributes are screened on the full subscription extract against the earlier churn field; issues per year and
tenure are screened on the main cohort against the 12-month target.

In [ ]:
def auc_alone(target, score):
    ok = ~pd.isna(score)
    a = roc_auc_score(np.asarray(target)[ok], np.asarray(score, dtype=float)[ok])
    return max(a, 1 - a)


earlier_target = subs.earlier_churn_event.astype(int)
table7 = pd.DataFrame([
    ('Last invoice amount', auc_alone(earlier_target, subs.last_invoice_paid)),
    ('Stored time-to-event', auc_alone(earlier_target, subs.earlier_churn_time_to_event)),
    ('Service issues per year', auc_alone(main.target_churn_12m, main.undated_issues_per_year)),
    ('Customer-service contact flag', auc_alone(earlier_target, subs.stats_has_service_contact.astype(float))),
    ('Tenure (for comparison)', auc_alone(main.target_churn_12m, main.tenure_days)),
], columns=['Attribute', 'ROC-AUC alone']).round(2)
table7

### 9.2 Adding the four undated attributes back (Figure 4)
The first-round models are trained twice on the same folds: once with the dated features only, and once with the four
undated attributes added together. Because the attributes are added together, the experiment shows their **combined**
effect; Table 7 above gives the strength of one of them, service issues per year, on its own.

A large gain from four undated attributes cannot come from customer behavior before the prediction date. It shows how
much of the earlier, near-perfect result came from leakage.

In [ ]:
yy, dated = first['1 July 2024']
# The manuscript (Section 2.9) also reports this run with n_trials=25: LightGBM AUC-PR 0.856.
_, leaky = first_round(cohorts['1 July 2024'], NUM + UNDATED, n_trials=15)

leak = pd.DataFrame({
    'AUC-PR, dated features': {k: average_precision_score(yy, dated[k]) for k in ['Logistic regression', 'LightGBM', 'CatBoost']},
    'AUC-PR, + undated': {k: average_precision_score(yy, leaky[k]) for k in ['Logistic regression', 'LightGBM', 'CatBoost']},
    'ROC-AUC, dated features': {k: roc_auc_score(yy, dated[k]) for k in ['Logistic regression', 'LightGBM', 'CatBoost']},
    'ROC-AUC, + undated': {k: roc_auc_score(yy, leaky[k]) for k in ['Logistic regression', 'LightGBM', 'CatBoost']},
}).round(3)
display(leak)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric, base in [(axes[0], 'AUC-PR', yy.mean()), (axes[1], 'ROC-AUC', 0.5)]:
    x = np.arange(3)
    ax.bar(x - 0.2, leak[f'{metric}, dated features'], 0.4, label='Dated features only', color='#4C72B0')
    ax.bar(x + 0.2, leak[f'{metric}, + undated'], 0.4, label='+ four undated attributes', color='#C44E52')
    ax.axhline(base, ls='--', color='grey')
    ax.set_xticks(x)
    ax.set_xticklabels(leak.index, fontsize=9)
    ax.set_title(metric)
    ax.set_ylim(0, 1)
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + 'figure4_leak_test.png', dpi=200)
plt.show()

## 10. Timing of cancellations within the 12-month window
How soon after the prediction date do churners actually leave? This matters for when retention contacts should start.

In [ ]:
days = main.loc[main.target_churn_12m == 1, 'outcome_days_to_event']
print(f'Median days from the prediction date to cancellation: {days.median():.0f}')
print(f'Cancelled within 90 days:  {(days <= 90).mean():.0%}')
print(f'Cancelled within 180 days: {(days <= 180).mean():.0%}')

## 11. Save the results
The tables are saved as CSV files in the `results/` folder. The out-of-fold scores are **not** saved, because they
are linked to individual subscriptions.

In [ ]:
table4.to_csv(OUT_DIR + 'table4_churn_by_group.csv', index=False)
table5.to_csv(OUT_DIR + 'table5_performance.csv', index=False)
bootstrap_table.to_csv(OUT_DIR + 'table5_bootstrap_intervals.csv', index=False)
pd.DataFrame(table6).T.to_csv(OUT_DIR + 'table6_robustness.csv')
table7.to_csv(OUT_DIR + 'table7_single_attribute_screen.csv', index=False)
leak.to_csv(OUT_DIR + 'figure4_leak_test.csv')
print('Saved to', OUT_DIR)